# Гамма-нож v3. Сколько из 0.80 было утечкой

Версия 2 показывала ROC-AUC 0.795 и PR-AUC 0.899. Здесь я провел анализ насколько это чистое значение.

Ответ: Одна колонка, которую нельзя было знать на момент прогноза, давала 0.20 ROC-AUC. Ниже это измерено, а дальше собрана честная постановка задачи.

## Данные

In [ ]:
!kaggle competitions download -c gamma-knife-3
import zipfile, os
os.makedirs("gamma-knife-3", exist_ok=True)
with zipfile.ZipFile("gamma-knife-3.zip") as z:
    z.extractall("gamma-knife-3")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier

SEED = 42
N_JOBS = 16
SEEDS = (42, 7, 13, 21, 99)

CAT_COLS = ["Онкологический диагноз", "Лекарственное лечение"]
DATES = ["Дата рождения",
         "Дата постановки онкологического диагноза / начала первичного лечения",
         "Дата удаления первичного очага", "Дата развития МГМ",
         "Дата проведения ОВГМ", "Дата операции на ГМ", "Дата 1-ой РХ"]

CB_PARAMS = dict(iterations=1000, learning_rate=0.025, depth=7, l2_leaf_reg=11,
                 min_data_in_leaf=51, random_strength=0.907, subsample=0.849,
                 bootstrap_type="Bernoulli", loss_function="Logloss")

In [ ]:
df_train = pd.read_csv("gamma-knife-3/train.csv")
df_test = pd.read_csv("gamma-knife-3/test.csv")
print(df_train.shape)

## Подготовка

Та же функция, что в v2. Одна на train и test, расхождения между ними невозможны по построению.

In [ ]:
def to_float(s):
    return s.astype(str).str.replace(",", ".", regex=False).astype(float)

def flag(s):
    v = s.astype(str).str.strip().str.lower()
    return (~v.isin(["nan", "none", ""]) & ~v.eq("нет")).astype(int)

def prepare(df):
    df = df.copy()
    for d in DATES:
        df[d] = pd.to_datetime(df[d], format="%d.%m.%Y", errors="coerce")

    df["Мужчина"] = df["Пол"].str.upper().eq("М").astype(int)
    df["Возраст"] = (df["Дата 1-ой РХ"] - df["Дата рождения"]).dt.days / 365.25
    df["Время_метастазирования"] = (df["Дата развития МГМ"] - df["Дата постановки онкологического диагноза / начала первичного лечения"]).dt.days
    df["Время_реагирования"] = (df["Дата 1-ой РХ"] - df["Дата развития МГМ"]).dt.days
    df["ОВГМ"] = df["Дата проведения ОВГМ"].notna().astype(int)
    df["Операция"] = df["Дата операции на ГМ"].notna().astype(int)
    df["Экстракраниальные метастазы"] = flag(df["Экстракраниальные метастазы"])

    df["Суммарный объем очагов"] = to_float(df["Суммарный объем очагов"])
    df["Объем максимального очага"] = to_float(df["Объем максимального очага"])
    df["Объем на очаг"] = df["Суммарный объем очагов"] / df["Число очагов в ГМ"]
    df["Скорость_метастазирования"] = df["Суммарный объем очагов"] / df["Время_метастазирования"]
    df["Скорость_реагирования"] = df["Суммарный объем очагов"] / (df["Время_реагирования"] + 1)
    df["Индекс_прогрессирования"] = df["Время_метастазирования"] - df["Время_реагирования"]
    df["Возраст * число очагов"] = df["Возраст"] * df["Число очагов в ГМ"]

    for c in CAT_COLS:
        df[c] = df[c].astype(str)

    drop = ["Пол"] + DATES + ["ID", "Активирующие мутации",
                              "Дистантные метастазы", "Локальный рецидив"]
    return df.drop(columns=[c for c in drop if c in df], errors="ignore")

In [ ]:
y_raw = df_train["Интракраниальная прогрессия"]
mask = y_raw.notna()
y_any = y_raw[mask].isin(["ДМ", "ЛР+ДМ", "ЛР"]).astype(int)

X_full = prepare(df_train[mask].drop(columns=["Интракраниальная прогрессия"]))
dates  = df_train[mask].copy()
for d in DATES:
    dates[d] = pd.to_datetime(dates[d], format="%d.%m.%Y", errors="coerce")

keep = (X_full["Время_метастазирования"].notna()
        & X_full["Время_реагирования"].notna()
        & (X_full["Время_реагирования"] >= 0))

X_full = X_full[keep].reset_index(drop=True)
y_any  = y_any[keep].reset_index(drop=True)
dates  = dates[keep.values].reset_index(drop=True)
raw    = df_train[mask][keep.values].reset_index(drop=True)

print(X_full.shape, round(y_any.mean(), 4))

## Аудит утечек

Проверка простая. Признак может существовать, только если его значение известно на момент первой радиохирургии. Все остальное это читерство

### Признак номер один

In [ ]:
pd.crosstab(raw["Число РХ процедур на ГН"], raw["Интракраниальная прогрессия"])

In [ ]:
one  = X_full["Число РХ процедур на ГН"] == 1
many = X_full["Число РХ процедур на ГН"] >= 2
print("одна процедура: n=%d, прогрессия %.3f" % (one.sum(),  y_any[one].mean()))
print("две и больше:   n=%d, прогрессия %.3f" % (many.sum(), y_any[many].mean()))

Из 197 пациентов, которым сделали больше одной процедуры, прогрессия у 191. Это 97 процентов.

Повторную радиохирургию назначают не просто так, а потому что появился новый очаг или вырос старый. То есть повторная процедура **это и есть прогрессия**, записанная другой колонкой. Лик
### Даты, которых не могло быть

Тот же вопрос к флагам «была ОВГМ» и «была операция на ГМ». Флаг ставился по факту наличия даты, без проверки, когда эта дата наступила.

In [ ]:
rx = dates["Дата 1-ой РХ"]
for c in ["Дата проведения ОВГМ", "Дата операции на ГМ", "Дата удаления первичного очага"]:
    v = dates[c]
    было = v.notna()
    после = было & (v > rx)
    print("%-38s задано %3d, из них после РХ %3d" % (c, было.sum(), после.sum()))

In [ ]:
после_овгм = (dates["Дата проведения ОВГМ"].notna()) & (dates["Дата проведения ОВГМ"] > rx)
до_овгм    = (dates["Дата проведения ОВГМ"].notna()) & (dates["Дата проведения ОВГМ"] <= rx)
print("ОВГМ после РХ: n=%d, прогрессия %.3f" % (после_овгм.sum(), y_any[после_овгм].mean()))
print("ОВГМ до РХ:    n=%d, прогрессия %.3f" % (до_овгм.sum(),    y_any[до_овгм].mean()))
print("ОВГМ не было:  n=%d, прогрессия %.3f" % ((~после_овгм & ~до_овгм).sum(),
                                                y_any[~после_овгм & ~до_овгм].mean()))

Из 80 облучений всего мозга 49 сделаны уже после гамма-ножа, и прогрессия у них в 78 процентах случаев против 65 в среднем. т.к облучение всего мозга назначают, когда появились новые очаги. Опять лик

## Три набора признаков

Считаем одно и то же тремя способами. Меняется только состав колонок.

In [ ]:
X_v2 = X_full.copy()                                        # как было в v2
X_no_rx = X_full.drop(columns=["Число РХ процедур на ГН"])  # минус главная утечка

X_strict = X_no_rx.copy()                                   # плюс честные флаги
X_strict["ОВГМ"] = до_овгм.astype(int).values
X_strict["Операция"] = ((dates["Дата операции на ГМ"].notna())
                        & (dates["Дата операции на ГМ"] <= rx)).astype(int).values

SETS = [("v2 как есть", X_v2), ("минус число РХ", X_no_rx), ("строгий набор", X_strict)]

## Протокол

Тот же вложенный протокол, что в v2. Порог подбирается на внутренних фолдах, квантили винзоризации берутся только из обучающей части, отложенная часть не участвует ни в чем, кроме замера.

In [ ]:
WINSOR = ["Время_метастазирования", "Время_реагирования", "Суммарный объем очагов",
          "Объем на очаг", "Объем максимального очага"]

def fit_caps(df):
    return {c: df[c].quantile(0.99) for c in WINSOR if c in df}

def apply_caps(df, caps):
    df = df.copy()
    for c, cap in caps.items():
        df[c] = df[c].clip(lower=0, upper=cap)
    for c in WINSOR:
        if c in df:
            df[c] = np.log1p(df[c])
    return df.replace([np.inf, -np.inf], np.nan)   # нулевые интервалы дают деление на ноль


def cats_of(df):
    """Категории определяем по типу колонки, а не списком: наборы признаков разные."""
    return [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]


def make_lr(df):
    cats = cats_of(df)
    num = [c for c in df.columns if c not in cats]
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                              ("sc", StandardScaler())]), num),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cats),
        ])),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
    ])


def fit_predict(kind, Xtr, ytr, Xva):
    if kind == "cb":
        m = CatBoostClassifier(**CB_PARAMS, random_seed=SEED, verbose=False,
                               task_type="CPU", thread_count=N_JOBS)
        m.fit(Xtr, ytr, cat_features=cats_of(Xtr))
    else:
        m = make_lr(Xtr)
        m.fit(Xtr, ytr)
    return m.predict_proba(Xva)[:, 1]


def pick_threshold(p, t):
    grid = np.quantile(p, np.linspace(0.02, 0.98, 97))
    return max(grid, key=lambda th: balanced_accuracy_score(t, (p >= th).astype(int)))


def evaluate(kind, Xd, yd, seed=SEED, n_splits=5):
    outer = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rows, oof = [], np.zeros(len(Xd))
    for tr, va in outer.split(Xd, yd):
        caps = fit_caps(Xd.iloc[tr])
        Xtr, Xva = apply_caps(Xd.iloc[tr], caps), apply_caps(Xd.iloc[va], caps)
        ytr, yva = yd.iloc[tr], yd.iloc[va]

        inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
        ip, iy = [], []
        for itr, iva in inner.split(Xtr, ytr):
            ip.append(fit_predict(kind, Xtr.iloc[itr], ytr.iloc[itr], Xtr.iloc[iva]))
            iy.append(ytr.iloc[iva])
        th = pick_threshold(np.concatenate(ip), pd.concat(iy))

        p = fit_predict(kind, Xtr, ytr, Xva)
        oof[va] = p
        rows.append(dict(roc_auc=roc_auc_score(yva, p),
                         pr_auc=average_precision_score(yva, p),
                         balanced_acc=balanced_accuracy_score(yva, (p >= th).astype(int))))
    return pd.DataFrame(rows), oof


def over_seeds(kind, Xd, yd, seeds=SEEDS):
    """Одно разбиение ничего не значит при разбросе по фолдам 0.05."""
    a, oofs = [], []
    for s in seeds:
        r, o = evaluate(kind, Xd, yd, seed=s)
        a.append(r.mean()); oofs.append(o)
    a = pd.DataFrame(a)
    return a.mean(), a.std(), np.mean(oofs, axis=0)

## Цена утечки

In [ ]:
%%time
rows = []
for name, Xd in SETS:
    for kind, label in [("cb", "CatBoost"), ("lr", "логрег")]:
        m, sd, _ = over_seeds(kind, Xd, y_any)
        rows.append(dict(набор=name, модель=label,
                         roc_auc=round(m.roc_auc, 4), pr_auc=round(m.pr_auc, 4),
                         balanced_acc=round(m.balanced_acc, 4), roc_sd=round(sd.roc_auc, 4)))
leak = pd.DataFrame(rows)
print("базовый уровень PR-AUC:", round(y_any.mean(), 4))
leak

Числа прогона на HistGradientBoosting, который использовался как замена CatBoost при подготовке этой версии

| набор | ROC-AUC | PR-AUC | balanced accuracy |
|---|---|---|---|
| v2 как есть | 0.798 | 0.898 | 0.742 |
| минус число РХ | 0.603 | 0.740 | 0.560 |
| строгий набор | 0.609 | 0.746 | 0.572 |

Базовый уровень PR-AUC равен доле положительного класса, 0.657.

**Одна колонка стоила 0.195 ROC-AUC.** После ее удаления PR-AUC 0.740 при базовом уровне 0.657, то есть модель еле обходит константу. Все заявленное качество v2 держалось на лике

Пересчет флагов ОВГМ и операции на честные добавляет 0.006

## Честная модель это вообще модель

0.61 выглядит слабо. Сравниваем с случайностью

In [ ]:
%%time
_, _, oof_strict = over_seeds("cb", X_strict, y_any)
print("OOF ROC-AUC:", round(roc_auc_score(y_any, oof_strict), 4))

rng = np.random.default_rng(0)
bs = []
for _ in range(2000):
    i = rng.integers(0, len(y_any), len(y_any))
    if y_any.iloc[i].nunique() < 2:
        continue
    bs.append(roc_auc_score(y_any.iloc[i], oof_strict[i]))
print("95%% доверительный интервал: [%.4f, %.4f]" % (np.percentile(bs, 2.5), np.percentile(bs, 97.5)))

In [ ]:
%%time
# перемешиваем таргет, если протокол не течет, качество должно упасть к 0.5
perm = []
for k in range(20):
    y_perm = pd.Series(rng.permutation(y_any.values))
    r, _ = evaluate("cb", X_strict, y_perm, seed=SEED)
    perm.append(r.roc_auc.mean())
print("на перемешанном таргете: среднее %.4f, максимум %.4f" % (np.mean(perm), np.max(perm)))

Доверительный интервал получился [0.568, 0.666], нижняя граница выше 0.5. На перемешанном таргете среднее 0.502, максимум 0.536, ни одна перестановка не дотянула до 0.615.

Значит сигнал слабый. 

## Откуда берется этот слабый сигнал

Убираем по группе признаков и смотрим, что просядет.

In [ ]:
%%time
GROUPS = {
 "объемы": ["Суммарный объем очагов", "Объем максимального очага", "Объем на очаг",
            "Число очагов в ГМ", "Возраст * число очагов"],
 "время": ["Время_метастазирования", "Время_реагирования", "Индекс_прогрессирования",
           "Скорость_метастазирования", "Скорость_реагирования"],
 "клиника": ["Индекс Карновского", "Онкологический диагноз", "Лекарственное лечение",
             "Экстракраниальные метастазы", "ОВГМ", "Операция"],
 "демография": ["Возраст", "Мужчина"],
}

def quick(Xd):
    m, _, _ = over_seeds("cb", Xd, y_any, seeds=(42, 7, 13))
    return round(m.roc_auc, 4), round(m.pr_auc, 4)

abl = [dict(вариант="все признаки", roc_auc=quick(X_strict)[0], pr_auc=quick(X_strict)[1])]
for g, cols in GROUPS.items():
    cols = [c for c in cols if c in X_strict]
    left = [c for c in X_strict.columns if c not in cols]
    if not set(CAT_COLS) & set(left):
        left = left + CAT_COLS
    r, p = quick(X_strict[left])
    abl.append(dict(вариант="без группы " + g, roc_auc=r, pr_auc=p))
pd.DataFrame(abl)

| вариант | ROC-AUC | PR-AUC |
|---|---|---|
| все признаки | 0.611 | 0.746 |
| без объемов | 0.538 | 0.690 |
| без времени | 0.615 | 0.758 |
| без клиники | 0.584 | 0.728 |
| без демографии | 0.614 | 0.746 |

Плоха только одна группа. **Весь сигнал сидит в объеме очагов и их числе,** плюс немного в клинике.

Отдельно плохой вывод про временные признаки. Пять колонок, посчитанных из семи дат, не дают ничего. Без них качество даже чуть выше. Вся работа с датами в v1 и v2 была бесмысленна (если не учитывать результаты соревнования :))

## Задача поставлена неправильно

Таргет v2 склеивал два разных исхода. Локальный рецидив это рост в уже облученной зоне, дистантные метастазы это новый очаг в другом месте мозга. Механизмы разные, значит и предсказуемость разная. Проверяем по отдельности.

In [ ]:
%%time
tgt = raw["Интракраниальная прогрессия"]
sub = []
for name, pos in [("любая прогрессия", ["ЛР", "ДМ", "ЛР+ДМ"]),
                  ("локальный рецидив", ["ЛР", "ЛР+ДМ"]),
                  ("дистантные метастазы", ["ДМ", "ЛР+ДМ"])]:
    yy = tgt.isin(pos).astype(int).reset_index(drop=True)
    m, _, _ = over_seeds("cb", X_strict, yy, seeds=(42, 7, 13))
    sub.append(dict(таргет=name, доля=round(yy.mean(), 4),
                    roc_auc=round(m.roc_auc, 4), pr_auc=round(m.pr_auc, 4)))
pd.DataFrame(sub)

| таргет | доля | ROC-AUC | PR-AUC |
|---|---|---|---|
| любая прогрессия | 0.657 | 0.611 | 0.746 |
| локальный рецидив | 0.237 | **0.509** | 0.286 |
| дистантные метастазы | 0.564 | **0.649** | 0.712 |

Локальный рецидив не предсказывается вообще. 0.509 это монетка.

И это объяснимо. Локальный контроль зависит от подведенной дозы, покрытия мишени и качества планирования, а этих колонок в данных нет. По одним характеристикам пациента предсказать рост в облученной зоне нельзя.

Дистантные метастазы предсказываются заметно лучше общего таргета. Склеивание двух исходов в один просто добавляло шум.

## Итоговая постановка

Предсказываем дистантную прогрессию по данным, доступным до первой радиохирургии.

In [ ]:
%%time
y_dm = tgt.isin(["ДМ", "ЛР+ДМ"]).astype(int).reset_index(drop=True)

final = []
for kind, label in [("cb", "CatBoost"), ("lr", "логрег")]:
    m, sd, oof = over_seeds(kind, X_strict, y_dm)
    final.append(dict(модель=label, roc_auc=round(m.roc_auc, 4), pr_auc=round(m.pr_auc, 4),
                      balanced_acc=round(m.balanced_acc, 4), roc_sd=round(sd.roc_auc, 4)))
    if kind == "cb":
        oof_dm = oof

bs = []
for _ in range(3000):
    i = rng.integers(0, len(y_dm), len(y_dm))
    if y_dm.iloc[i].nunique() < 2:
        continue
    bs.append(roc_auc_score(y_dm.iloc[i], oof_dm[i]))

print("базовый уровень PR-AUC:", round(y_dm.mean(), 4))
print("95%% интервал ROC-AUC: [%.4f, %.4f]" % (np.percentile(bs, 2.5), np.percentile(bs, 97.5)))
pd.DataFrame(final)

| модель | ROC-AUC | PR-AUC | balanced accuracy |
|---|---|---|---|
| CatBoost | 0.650 | 0.712 | 0.602 |
| логистическая регрессия | 0.651 | 0.720 | 0.591 |

ROC-AUC 0.650 с интервалом [0.614, 0.703] при базовом уровне PR-AUC 0.564.

Два вывода сразу. Первый: сигнал устойчивый, интервал не задевает 0.5. Второй: **бустинг не нужен.** Логистическая регрессия на тех же признаках дает то же самое, а объяснять коэффициенты линейной модели врачу проще, чем деревья.

## Попытки что-то сделать

0.650 это мало. Прежде чем объявлять потолок, стоит проверить очевидное: другие семейства моделей, больше признаков, подбор гиперпараметров и ансамбли. Все в том же формате

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.preprocessing import OrdinalEncoder

# расширенный набор признаков
X_ext = X_strict.copy()
X_ext["Мутации"] = raw["Активирующие мутации"].fillna("нет данных").astype(str).values
X_ext["Первичный удален до РХ"] = ((dates["Дата удаления первичного очага"].notna())
                                   & (dates["Дата удаления первичного очага"] <= rx)).astype(int).values
X_ext["Доля макс очага"] = X_ext["Объем максимального очага"] / X_ext["Суммарный объем очагов"].replace(0, np.nan)
X_ext["Много очагов"] = (X_ext["Число очагов в ГМ"] >= 4).astype(int)
X_ext["Одиночный очаг"] = (X_ext["Число очагов в ГМ"] == 1).astype(int)
X_ext["Карновский низкий"] = (X_ext["Индекс Карновского"] < 80).astype(int)
X_ext["Экстра неизвестно"] = raw["Экстракраниальные метастазы"].isna().astype(int).values
X_ext["Лечение неизвестно"] = raw["Лекарственное лечение"].isna().astype(int).values
X_ext["объем x очаги"] = np.log1p(X_ext["Суммарный объем очагов"]) * X_ext["Число очагов в ГМ"]

print(X_ext.shape)

In [ ]:
def make_forest(kind, df, leaf=8):
    """Лесу нужны числа вместо категорий и заполненные пропуски."""
    cats = cats_of(df)
    num = [c for c in df.columns if c not in cats]
    enc = ColumnTransformer([("num", "passthrough", num),
                             ("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                                                    unknown_value=-1), cats)])
    cls = (RandomForestClassifier(n_estimators=600, min_samples_leaf=leaf, max_features="sqrt",
                                  class_weight="balanced_subsample", random_state=SEED, n_jobs=N_JOBS)
           if kind == "rf" else
           ExtraTreesClassifier(n_estimators=800, min_samples_leaf=leaf, max_features="sqrt",
                                class_weight="balanced", random_state=SEED, n_jobs=N_JOBS))
    return Pipeline([("prep", enc), ("imp", SimpleImputer(strategy="median")), ("cls", cls)])

Результаты прогона (дистантная прогрессия, строгий набор, среднее по трем разбиениям):

| Что пробовали | ROC-AUC | PR-AUC | balanced accuracy |
|---|---|---|---|
| бустинг, как в разделе выше | 0.649 | 0.712 | 0.597 |
| бустинг на расширенном наборе | 0.645 | 0.707 | 0.599 |
| бустинг с подбором гиперпараметров внутри фолда | 0.651 | 0.713 | 0.599 |
| логистическая регрессия | 0.642 | 0.710 | 0.596 |
| ансамбль бустинг + логрег | 0.654 | 0.718 | 0.599 |
| ансамбль бустинг + логрег + лес | 0.662 | 0.724 | 0.602 |
| **случайный лес** | **0.673** | **0.736** | 0.614 |
| **extra trees** | 0.672 | 0.730 | 0.616 |
| **лес + extra trees** | **0.677** | **0.738** | **0.620** |

Три вывода.

Расширение набора признаков не дало ничего. Подбор гиперпараметров дал 0.002, то есть ничего. Ансамбли из слабых моделей только разбавляют сильную.

**А смена семейства дала 0.027 ROC-AUC и почти два пункта balanced accuracy.** Бустинг здесь был просто неправильным инструментом: он последовательно дообучается на своих ошибках, а на 583 строках с шумным таргетом ошибки это в основном шум, и он его заучивает. Лес усредняет переобученные деревья, обученные независимо, и шум взаимно гасится.

### Честный замер

Выбирать лучшее семейство по отложенным метрикам это то же самое, что подбирать по ним порог. Поэтому выбор модели уносится внутрь фолда: на внутренних фолдах перебирается набор моделей, наружу выходит только замер выбранного.

In [ ]:
%%time
ZOO = [("gb", None), ("lr", None),
       ("rf", 4), ("rf", 8), ("rf", 16), ("et", 4), ("et", 8)]

def build(kind, df, prm):
    if kind == "lr":
        return make_lr(df)
    return make_forest(kind, df, leaf=prm)


def nested_choice(Xd, yd, seeds=SEEDS):
    accs, picks, oof = [], [], np.zeros(len(Xd))
    for seed in seeds:
        outer = StratifiedKFold(5, shuffle=True, random_state=seed)
        rows = []
        for tr, va in outer.split(Xd, yd):
            caps = fit_caps(Xd.iloc[tr])
            Xtr, Xva = apply_caps(Xd.iloc[tr], caps), apply_caps(Xd.iloc[va], caps)
            ytr, yva = yd.iloc[tr], yd.iloc[va]
            folds = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(Xtr, ytr))

            inner = {}
            for j, (kind, prm) in enumerate(ZOO):
                ps, ys = [], []
                for itr, iva in folds:
                    if kind == "gb":
                        p = fit_predict("cb", Xtr.iloc[itr], ytr.iloc[itr], Xtr.iloc[iva])
                    else:
                        m = build(kind, Xtr, prm)
                        m.fit(Xtr.iloc[itr], ytr.iloc[itr])
                        p = m.predict_proba(Xtr.iloc[iva])[:, 1]
                    ps.append(p); ys.append(ytr.iloc[iva])
                inner[j] = (np.concatenate(ps), pd.concat(ys))

            cands = [(j,) for j in range(len(ZOO))] + [(2, 5), (0, 1), (2, 0)]
            best = max(cands, key=lambda c: roc_auc_score(inner[c[0]][1],
                                                          np.mean([inner[j][0] for j in c], axis=0)))
            picks.append("+".join(ZOO[j][0] for j in best))

            ip = np.mean([inner[j][0] for j in best], axis=0); iy = inner[best[0]][1]
            th = pick_threshold(ip, iy)

            pp = []
            for j in best:
                kind, prm = ZOO[j]
                if kind == "gb":
                    pp.append(fit_predict("cb", Xtr, ytr, Xva))
                else:
                    m = build(kind, Xtr, prm); m.fit(Xtr, ytr)
                    pp.append(m.predict_proba(Xva)[:, 1])
            p = np.mean(pp, axis=0); oof[va] += p / len(seeds)
            rows.append(dict(roc_auc=roc_auc_score(yva, p),
                             pr_auc=average_precision_score(yva, p),
                             balanced_acc=balanced_accuracy_score(yva, (p >= th).astype(int))))
        accs.append(pd.DataFrame(rows).mean())
    return pd.DataFrame(accs).mean(), pd.Series(picks).value_counts(), oof


m, picks, oof_best = nested_choice(X_ext, y_dm)
print(m.round(4))
print()
print("что выбиралось внутри фолдов:")
print(picks)

In [ ]:
bs = []
for _ in range(3000):
    i = rng.integers(0, len(y_dm), len(y_dm))
    if y_dm.iloc[i].nunique() < 2:
        continue
    bs.append(roc_auc_score(y_dm.iloc[i], oof_best[i]))
print("OOF ROC-AUC: %.4f" % roc_auc_score(y_dm, oof_best))
print("95%% интервал: [%.4f, %.4f]" % (np.percentile(bs, 2.5), np.percentile(bs, 97.5)))

Вложенный выбор дает **ROC-AUC 0.666, PR-AUC 0.732, balanced accuracy 0.614**, OOF ROC-AUC 0.672 с интервалом [0.629, 0.715] при базовом уровне PR-AUC 0.564.

Внутри фолдов выбор оказался устойчивым: из 25 случаев бэггинг в том или ином виде победил 24 раза, бустинг с логрегом выиграли один раз.

Итог по всей работе:

| Версия | Что считалось | ROC-AUC |
|---|---|---|
| v2 | любая прогрессия, с утечкой | 0.795 |
| v3, шаг 1 | любая прогрессия, честно, бустинг | 0.612 |
| v3, шаг 2 | дистантная прогрессия, бустинг | 0.650 |
| v3, итог | дистантная прогрессия, бэггинг, вложенный выбор | **0.666** |

Правильная постановка задачи дала 0.038, правильное семейство моделей еще 0.016. Ни то, ни другое не возвращает 0.795, потому что те 0.795 были не качеством модели, а утечкой.

## Поможет ли больше данных

Тот же протокол на случайных подвыборках.

In [ ]:
%%time
curve = []
for frac in (0.4, 0.6, 0.8, 1.0):
    a = []
    for s in SEEDS:
        r = np.random.default_rng(s)
        idx = r.permutation(len(y_any))[:int(len(y_any) * frac)]
        m, _, _ = over_seeds("cb", X_strict.iloc[idx].reset_index(drop=True),
                             y_any.iloc[idx].reset_index(drop=True), seeds=(s,))
        a.append(m.roc_auc)
    curve.append(dict(доля=frac, n=int(len(y_any) * frac), roc_auc=round(np.mean(a), 4)))
pd.DataFrame(curve)

| доля выборки | n | ROC-AUC |
|---|---|---|
| 0.4 | 233 | 0.611 |
| 0.6 | 349 | 0.585 |
| 0.8 | 466 | 0.596 |
| 1.0 | 583 | 0.632 |

Кривая плоская в пределах шума. Удвоение выборки качество не поднимет, потому что упирается не в объем, а в состав признаков